# Semantic Similarity Computation with SBERT

This notebook performs the following steps:
1. Loading parallel corpora from `.txt` files (one file per language/dialect)
2. Building balanced subsets: parallel, semi-parallel and non-parallel pairs
3. Computing semantic similarity scores (STS) using `paraphrase-multilingual-MiniLM-L12-v2`
4. Computing complementary features (length score, Jaccard indices, phonetic similarity)
5. Exporting the annotated dataset as `.csv` for classifier training

---

### Expected file format

Each file contains **one sentence per line** with **no trailing newline required** on the last line.
The source French file and the target regional language file must have the **same number of non-empty lines**.

Two corpus layouts are supported:

#### Layout A — Shared French source (LIMSI Atlas)
One single French file serves as the source for all regional language targets.
All target files must have the same number of sentences as the shared French file.

```
corpus/
├── fr.txt                 ← single shared French source
├── gsw_strasbourg.txt     ← Alsatian (Strasbourg dialect)
├── gsw_mulhouse.txt       ← Alsatian (Mulhouse dialect)
├── co_ajaccio.txt         ← Corsican (Ajaccio)
├── oc_beyssac.txt         ← Occitan (Noc Beyssac variety)
└── oc_gascon.txt          ← Occitan (Gascon variety)
```

#### Layout B — Separate French source per pair (ParCoLab / Parabole)
Each regional language file has its own paired French source file.

```
corpus/
├── fr_gsw_fr.txt          ← French source for Alsatian
├── fr_gsw_strasbourg.txt  ← Alsatian target
├── fr_co_fr.txt           ← French source for Corsican
├── fr_co_co.txt           ← Corsican target
└── ...
```

> Both layouts can be combined in `FILE_PAIRS` — simply point each entry to the correct French source file.

## 1. Installation and imports

In [ ]:
!pip install sentence-transformers jellyfish pandas numpy scikit-learn tqdm matplotlib

In [ ]:
import os
import random
import unicodedata
import re
import string

import pandas as pd
import numpy as np
import jellyfish
import matplotlib.pyplot as plt

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

# Reproducibility
random.seed(42)
np.random.seed(42)

print("Imports successful.")

## 2. Configuration

Each entry in `FILE_PAIRS` maps:
- a **French source file** (can be shared across multiple targets)
- a **regional language target file**
- a **label** identifying the language pair and dialect

The LIMSI Atlas uses a **shared French source**: the same `fr.txt` is paired with each dialect file.
ParCoLab and the Parabole use **separate French source files** per pair.

In [ ]:
# ============================================================
# CONFIGURATION - ADAPT TO YOUR FILE STRUCTURE
# ============================================================

# Directory containing the .txt files
CORPUS_DIR = "corpus/"

# List of aligned file pairs: (french_source_file, regional_target_file, label)
#
# LIMSI Atlas → shared French source (same fr.txt for all targets)
# ParCoLab / Parabole → separate French source per pair
FILE_PAIRS = [
    # --- LIMSI Atlas (shared French source) ---
    # Alsatian dialects
    ("Fable_esope_la_bise_et_le_soleil_fr_paris.txt",  "Fable_esope_la_bise_et_le_soleil_gsw_strasbourg.txt", "fr-gsw-strasbourg"),
    # Add other Alsatian dialects here with the same French source:
    # ("Fable_esope_la_bise_et_le_soleil_fr_paris.txt",  "gsw_mulhouse.txt",     "fr-gsw-mulhouse"),
    # Corsican
    ("Fable_esope_la_bise_et_le_soleil_fr_paris.txt",  "Fable_esope_la_bise_et_le_soleil_co_ajaccio.txt",    "fr-co-ajaccio"),
    # Occitan varieties
    ("Fable_esope_la_bise_et_le_soleil_fr_paris.txt",  "Fable_esope_la_bise_et_le_soleil_oc_noc_beyssac.txt","fr-oc-noc-beyssac"),
    # Add other Occitan varieties here:
    # ("Fable_esope_la_bise_et_le_soleil_fr_paris.txt",  "oc_gascon.txt",        "fr-oc-gascon"),

    # --- ParCoLab (separate French source per pair) ---
    # ("fr_gsw_fr.txt",   "fr_gsw_target.txt",  "fr-gsw-parcolab"),
    # ("fr_co_fr.txt",    "fr_co_target.txt",   "fr-co-parcolab"),
    # ("fr_oc_fr.txt",    "fr_oc_target.txt",   "fr-oc-parcolab"),

    # --- Parabole du fils prodigue (separate French source) ---
    # ("parabole_fr.txt", "parabole_oc.txt",    "fr-oc-parabole"),
]

# SBERT model
SBERT_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"

# Number of pairs per subset category
# None = use all available parallel pairs
# Integer = sample exactly that many pairs per category
SUBSET_SIZE = None

# Output file
OUTPUT_FILE = "similarity_dataset.csv"

print(f"Configuration loaded: {len(FILE_PAIRS)} file pairs defined.")

## 3. Loading parallel corpora

Files are read line by line. Empty lines are skipped automatically.
The last line does not need a trailing newline — both formats are handled correctly.

In [ ]:
def read_sentences(filepath):
    """
    Read a .txt file and return a list of non-empty stripped sentences.
    Compatible with files that have or do not have a trailing newline.
    """
    with open(filepath, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def load_pair(fr_path, lr_path, label):
    """
    Load two aligned .txt files.
    Returns a list of tuples (fr_sentence, lr_sentence, label).
    If sentence counts differ, truncates to the minimum and prints a warning.
    """
    fr_sentences = read_sentences(fr_path)
    lr_sentences = read_sentences(lr_path)

    if len(fr_sentences) != len(lr_sentences):
        n = min(len(fr_sentences), len(lr_sentences))
        print(f"  ⚠️  [{label}] Misalignment: {len(fr_sentences)} FR / "
              f"{len(lr_sentences)} LR → truncated to {n}")
        fr_sentences, lr_sentences = fr_sentences[:n], lr_sentences[:n]

    return [(fr, lr, label) for fr, lr in zip(fr_sentences, lr_sentences)]


print("Loading parallel corpora...\n")
parallel_pairs = []

for fr_file, lr_file, label in FILE_PAIRS:
    fr_path = os.path.join(CORPUS_DIR, fr_file)
    lr_path = os.path.join(CORPUS_DIR, lr_file)

    if not os.path.exists(fr_path):
        print(f"  ✖ [{label}] File not found: {fr_path}")
        continue
    if not os.path.exists(lr_path):
        print(f"  ✖ [{label}] File not found: {lr_path}")
        continue

    pairs = load_pair(fr_path, lr_path, label)
    parallel_pairs.extend(pairs)
    print(f"  ✔ [{label}] {len(pairs)} pairs loaded")

print(f"\nTotal parallel pairs loaded: {len(parallel_pairs)}")

# Warn if the corpus is very small (typical for single LIMSI fable files)
if len(parallel_pairs) < 50:
    print(f"\n  ℹ️  Small corpus ({len(parallel_pairs)} pairs). "
          f"Consider adding more dialect files to FILE_PAIRS for robust training.")

## 4. Corpus inspection

Before building subsets, display the loaded pairs to verify alignment quality.

In [ ]:
def inspect_corpus(pairs, n_per_label=2):
    """
    Display a sample of loaded pairs grouped by language label.
    Useful for visually verifying alignment quality.
    """
    from collections import defaultdict
    by_label = defaultdict(list)
    for fr, lr, lbl in pairs:
        by_label[lbl].append((fr, lr))

    for label, examples in by_label.items():
        print(f"\n{'─'*70}")
        print(f" {label}  ({len(examples)} pairs total)")
        print('─'*70)
        for fr, lr in examples[:n_per_label]:
            print(f"  FR: {fr[:100]}")
            print(f"  LR: {lr[:100]}")
            print()

inspect_corpus(parallel_pairs)

## 5. Building subsets

Three equally-sized subsets are built from the parallel pairs:

| Subset | Construction |
|--------|--------------|
| **Parallel** | Pairs directly taken from the aligned files |
| **Semi-parallel** | Pairs where the target sentence is truncated (after comma/semicolon or at mid-length) |
| **Non-parallel** | Pairs built by randomly shuffling FR ↔ LR sentences at different positions |

> **Note on small corpora**: With only 5 sentences per dialect file (LIMSI fable), the subsets will be very small. Adding more dialect files in `FILE_PAIRS` directly increases the subset size.

In [ ]:
def build_parallel_subset(pairs, size=None):
    """
    Build the parallel subset.
    If size is set, randomly samples that many pairs from the full list.
    """
    if size and size < len(pairs):
        pairs = random.sample(pairs, size)
    return [(fr, lr, lbl, "parallel") for fr, lr, lbl in pairs]


def build_semi_parallel_subset(pairs, size):
    """
    Build the semi-parallel subset by truncating the target sentence.

    Strategy 1 (preferred): split after the first comma, semicolon or colon
                            and keep the first segment.
    Strategy 2 (fallback):  keep only the first half of words when no
                            internal punctuation is available.

    This mimics the partial translations and dialectal omissions frequently
    observed in LIMSI Atlas transcriptions.
    """
    sample = random.sample(pairs, min(size, len(pairs)))
    subset = []

    for fr, lr, lbl in sample:
        segments = re.split(r'[,;:]', lr, maxsplit=1)
        if len(segments) > 1 and len(segments[0].strip()) > 3:
            truncated = segments[0].strip()
        else:
            words = lr.split()
            truncated = " ".join(words[:max(1, len(words) // 2)])
        subset.append((fr, truncated, lbl, "semi-parallel"))

    return subset


def build_non_parallel_subset(pairs, size):
    """
    Build the non-parallel subset by randomly pairing FR and LR sentences
    at different positions.

    Checks that the generated pair does not already exist in the parallel
    set to avoid introducing false negatives.

    When the corpus is small (e.g. a single LIMSI fable with 5 sentences),
    the number of unique non-parallel pairs is limited: at most n*(n-1)
    where n is the number of sentences. A warning is printed if the target
    size cannot be reached.
    """
    fr_sentences   = [p[0] for p in pairs]
    lr_sentences   = [p[1] for p in pairs]
    labels         = [p[2] for p in pairs]
    existing_pairs = set((p[0], p[1]) for p in pairs)

    subset   = []
    attempts = 0

    while len(subset) < size and attempts < size * 20:
        i = random.randint(0, len(fr_sentences) - 1)
        j = random.randint(0, len(lr_sentences) - 1)
        attempts += 1
        if i != j and (fr_sentences[i], lr_sentences[j]) not in existing_pairs:
            subset.append((fr_sentences[i], lr_sentences[j], labels[i], "non-parallel"))
            existing_pairs.add((fr_sentences[i], lr_sentences[j]))

    if len(subset) < size:
        print(f"  ⚠️  Only {len(subset)}/{size} non-parallel pairs generated. "
              f"This is expected with small corpora — add more dialect files to increase coverage.")
    return subset


# -------------------------------------------------------
# Build the three subsets
# -------------------------------------------------------
target_size = min(SUBSET_SIZE or len(parallel_pairs), len(parallel_pairs))
print(f"Target size per subset: {target_size}\n")

subset_parallel      = build_parallel_subset(parallel_pairs, target_size)
print(f"  ✔ Parallel subset:      {len(subset_parallel)} pairs")

subset_semi          = build_semi_parallel_subset(parallel_pairs, target_size)
print(f"  ✔ Semi-parallel subset: {len(subset_semi)} pairs")

subset_non_parallel  = build_non_parallel_subset(parallel_pairs, target_size)
print(f"  ✔ Non-parallel subset:  {len(subset_non_parallel)} pairs")

# Merge and shuffle the full dataset
dataset = subset_parallel + subset_semi + subset_non_parallel
random.shuffle(dataset)
print(f"\nFull shuffled dataset: {len(dataset)} pairs")

## 6. Feature computation functions

In [ ]:
def normalize(text):
    """
    Normalize a string: lowercase, remove accents and punctuation.
    Applied before all Jaccard index computations to reduce orthographic
    noise, which is especially important for unstandardized regional languages.
    """
    text = text.lower()
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    return text.translate(str.maketrans("", "", string.punctuation)).strip()


def length_score(s1, s2):
    """
    Compute normalized length asymmetry between two sentences.
    score = |len(s1) - len(s2)| / max(len(s1), len(s2))
    0 = identical word count (favorable to parallelism)
    1 = very asymmetric (unfavorable)
    """
    n1, n2  = len(s1.split()), len(s2.split())
    max_len = max(n1, n2)
    return 0.0 if max_len == 0 else abs(n1 - n2) / max_len


def char_ngrams(text, n):
    """Return the set of character n-grams for a normalized string."""
    t = normalize(text)
    return set(t[i:i+n] for i in range(len(t) - n + 1))


def jaccard_index(set_a, set_b):
    """Compute the Jaccard index: |A∩B| / |A∪B|."""
    if not set_a and not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)


def jaccard_ngrams(s1, s2, n):
    """
    Compute the Jaccard index on character n-grams of two sentences.
    Captures surface-level lexical overlap including shared roots between
    French and Romance regional languages (Occitan, Corsican).
    """
    return jaccard_index(char_ngrams(s1, n), char_ngrams(s2, n))


def metaphone_codes(text):
    """
    Encode each word using the Metaphone algorithm (jellyfish library).
    Returns the set of non-empty phonetic codes.
    Words that cannot be encoded are silently skipped.

    Particularly useful for Occitan and Corsican which share phonetic
    roots with French that are more visible in pronunciation than in spelling.
    """
    codes = set()
    for word in normalize(text).split():
        try:
            code = jellyfish.metaphone(word)
            if code:
                codes.add(code)
        except Exception:
            pass
    return codes


def phonetic_jaccard(s1, s2):
    """Compute the Jaccard index on Metaphone phonetic representations."""
    return jaccard_index(metaphone_codes(s1), metaphone_codes(s2))


print("Feature functions defined.")

# -------------------------------------------------------
# Sanity check on the actual loaded data
# -------------------------------------------------------
if parallel_pairs:
    print("\n--- Sanity check on first loaded pair ---")
    fr_ex, lr_ex, lbl_ex = parallel_pairs[0]
    print(f"Label : {lbl_ex}")
    print(f"FR    : {fr_ex[:90]}")
    print(f"LR    : {lr_ex[:90]}")
    print(f"  Length score   : {length_score(fr_ex, lr_ex):.4f}")
    print(f"  Jaccard-2      : {jaccard_ngrams(fr_ex, lr_ex, 2):.4f}")
    print(f"  Jaccard-3      : {jaccard_ngrams(fr_ex, lr_ex, 3):.4f}")
    print(f"  Jaccard-4      : {jaccard_ngrams(fr_ex, lr_ex, 4):.4f}")
    print(f"  Phonetic       : {phonetic_jaccard(fr_ex, lr_ex):.4f}")

## 7. SBERT score computation

In [ ]:
print(f"Loading SBERT model: {SBERT_MODEL} ...")
model = SentenceTransformer(SBERT_MODEL)
print("Model loaded.")

In [ ]:
def compute_sbert_scores(sentences_1, sentences_2, model, batch_size=64):
    """
    Compute SBERT cosine similarity scores for a list of sentence pairs.
    Processes in batches for memory efficiency.
    Scores are clipped to [0, 1] since cosine similarity can be slightly negative.
    Returns a list of float scores.
    """
    scores = []
    for i in tqdm(range(0, len(sentences_1), batch_size), desc="SBERT"):
        batch_1 = sentences_1[i:i+batch_size]
        batch_2 = sentences_2[i:i+batch_size]
        emb_1   = model.encode(batch_1, convert_to_numpy=True, show_progress_bar=False)
        emb_2   = model.encode(batch_2, convert_to_numpy=True, show_progress_bar=False)
        for v1, v2 in zip(emb_1, emb_2):
            scores.append(float(max(0.0, cosine_similarity([v1], [v2])[0][0])))
    return scores


all_fr_sentences = [pair[0] for pair in dataset]
all_lr_sentences = [pair[1] for pair in dataset]

print(f"Computing SBERT scores on {len(dataset)} pairs...")
sbert_scores = compute_sbert_scores(all_fr_sentences, all_lr_sentences, model)
print("Done.")

## 8. Building the annotated DataFrame

In [ ]:
print("Computing complementary features...")

rows = []
for idx, (fr, lr, lang_label, label_class) in enumerate(tqdm(dataset, desc="Features")):
    rows.append({
        "fr_sentence":         fr,
        "lr_sentence":         lr,
        "language_pair":       lang_label,
        "class":               label_class,
        # Binary label: 1 = parallel or semi-parallel | 0 = non-parallel
        "binary_class":        0 if label_class == "non-parallel" else 1,
        # SBERT cosine similarity score
        "sbert_score":         sbert_scores[idx],
        # Normalized length asymmetry
        "length_score":        length_score(fr, lr),
        # Character n-gram Jaccard indices
        "jaccard_bigrams":     jaccard_ngrams(fr, lr, 2),
        "jaccard_trigrams":    jaccard_ngrams(fr, lr, 3),
        "jaccard_quadrigrams": jaccard_ngrams(fr, lr, 4),
        # Phonetic Jaccard (Metaphone encoding)
        "phonetic_jaccard":    phonetic_jaccard(fr, lr),
    })

df = pd.DataFrame(rows)
print(f"DataFrame: {len(df)} rows x {len(df.columns)} columns")
df.head()

## 9. Descriptive analysis by subset

In [ ]:
numeric_features = [
    "sbert_score", "length_score",
    "jaccard_bigrams", "jaccard_trigrams",
    "jaccard_quadrigrams", "phonetic_jaccard"
]

print("=== Descriptive statistics by class ===")
print(df.groupby("class")[numeric_features].agg(["mean", "std"]).round(4).to_string())

print("\n=== Mean scores by language pair ===")
print(df.groupby("language_pair")[numeric_features].mean().round(4).to_string())

In [ ]:
# Skip visualizations if the dataset is too small to be meaningful
if len(df) < 6:
    print("Dataset too small for meaningful plots. Add more dialect files to FILE_PAIRS.")
else:
    CLASS_COLORS = {
        "parallel":      "#2ecc71",
        "semi-parallel": "#f39c12",
        "non-parallel":  "#e74c3c"
    }

    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()

    for i, feature in enumerate(numeric_features):
        for class_label, color in CLASS_COLORS.items():
            subset = df[df["class"] == class_label][feature]
            if not subset.empty:
                axes[i].hist(subset, bins=max(5, len(subset)//2),
                             alpha=0.6, label=class_label, color=color)
        axes[i].set_title(feature.replace("_", " ").title())
        axes[i].set_xlabel("Score")
        axes[i].set_ylabel("Frequency")
        axes[i].legend(fontsize=8)

    plt.suptitle("Feature distributions by pair category", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig("feature_distributions.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Figure saved: feature_distributions.png")

In [ ]:
if len(df) >= 6:
    pivot = df.groupby(["language_pair", "class"])["sbert_score"].mean().unstack()

    fig, ax = plt.subplots(figsize=(12, 5))
    pivot.plot(
        kind="bar", ax=ax,
        color=[CLASS_COLORS.get(c, "grey") for c in pivot.columns],
        rot=30
    )
    ax.set_title("Mean SBERT score by language pair and category")
    ax.set_xlabel("Language pair")
    ax.set_ylabel("Mean SBERT score")
    ax.legend(title="Class")
    plt.tight_layout()
    plt.savefig("sbert_by_language.png", dpi=150)
    plt.show()
    print("Figure saved: sbert_by_language.png")

## 10. Sample pairs per subset

In [ ]:
def display_examples(df, class_label, n=3):
    """Display n random examples for a given class label."""
    print(f"\n{'='*70}")
    print(f" Examples: {class_label.upper()}")
    print('='*70)
    subset = df[df["class"] == class_label]
    if subset.empty:
        print("  (no pairs in this category)")
        return
    for _, row in subset.sample(min(n, len(subset))).iterrows():
        print(f"  FR             : {row['fr_sentence'][:95]}")
        print(f"  LR             : {row['lr_sentence'][:95]}")
        print(f"  Language pair  : {row['language_pair']}")
        print(f"  SBERT          : {row['sbert_score']:.4f}")
        print(f"  Length score   : {row['length_score']:.4f}")
        print(f"  Jaccard-3      : {row['jaccard_trigrams']:.4f}")
        print(f"  Phonetic       : {row['phonetic_jaccard']:.4f}")
        print()

display_examples(df, "parallel")
display_examples(df, "semi-parallel")
display_examples(df, "non-parallel")

## 11. Export annotated dataset

In [ ]:
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"Dataset exported: {OUTPUT_FILE}")
print(f"\nClass distribution:")
print(df["class"].value_counts().to_string())
print(f"\nLanguage pair distribution:")
print(df["language_pair"].value_counts().to_string())

feature_columns = [
    c for c in df.columns
    if c not in ["fr_sentence", "lr_sentence", "language_pair", "class"]
]
print(f"\nFeature columns available for the classifier: {feature_columns}")
print("\n✔ Ready for the logistic regression classifier notebook.")